# Fine-Tuning Qwen2.5-3B for Structured Job-Posting Extraction (QLoRA)

Fine-tunes **Qwen2.5-3B-Instruct** to extract structured JSON — `{required_skills, tech_stack, seniority, avg_comp_range}` — from raw AI-engineer job postings, using **QLoRA** (4-bit quantized base + trainable LoRA adapters).

**Pipeline:** setup → format supervised examples → attach LoRA adapters → fine-tune → evaluate base-vs-tuned on a held-out test set → publish the adapter to the Hugging Face Hub.

Runs on a single free Colab **T4** GPU. The labeled dataset, extraction prompt (`src/prompts.py`), schema (`src/schema.py`), and field-level metric (`src/evaluation.py`) live in the project repo, which is cloned below.


## 1. Environment setup

Install the fine-tuning stack, then load the base model in 4-bit (the "Q" in QLoRA — it shrinks the frozen ~3B base from ~6 GB to ~2 GB so it fits the T4 with room to train adapters on top).

> ⚠️ **After running the install cell, restart the runtime** (Runtime → Restart session) before running anything else — Colab pre-imports `torch`, and an in-place upgrade leaves it in a broken half-state until the kernel restarts.


In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl

In [ ]:
%cd /content
!git clone -q https://github.com/tkatz123/Fine_Tuning_LLM_Microservice.git
%cd Fine_Tuning_LLM_Microservice
from google.colab import drive; drive.mount('/content/drive')

import torch, json
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from src.prompts import EXTRACTION_PROMPT

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
print("base loaded ✅")

## 2. Format the supervised fine-tuning (SFT) examples

Each training example is the full chat — system prompt + posting + the **gold JSON as the assistant turn** — so the model learns to reproduce clean, schema-valid output. The posting text (input) lives in the raw CSV and is joined to its gold label by `id`.


In [ ]:
df = pd.read_csv("data/raw/AI_Engineer_Job_Data.csv"); df.columns=[c.lstrip("\ufeff") for c in df.columns]

def format_example(rec):
    jd = df.loc[rec["id"], "job_description"]
    target = {"required_skills": rec["required_skills"], "tech_stack": rec["tech_stack"],
              "seniority": rec["seniority"], "avg_comp_range": rec["avg_comp_range"]}
    messages = [{"role":"system","content":EXTRACTION_PROMPT},
                {"role":"user","content":jd},
                {"role":"assistant","content":json.dumps(target)}]
    return tokenizer.apply_chat_template(messages, tokenize=False)

from datasets import Dataset
train = [json.loads(l) for l in open("data/processed/train.jsonl") if l.strip()]
val   = [json.loads(l) for l in open("data/processed/val.jsonl")   if l.strip()]
train_ds = Dataset.from_dict({"text":[format_example(r) for r in train]})
val_ds   = Dataset.from_dict({"text":[format_example(r) for r in val]})
print(f"train {len(train_ds)}  val {len(val_ds)}")

## 3. Attach LoRA adapters

Freeze the 4-bit base and train only small low-rank adapters (`ΔW = B·A`, rank 16). `print_trainable_parameters()` confirms we train **<1%** of the model's parameters — the reason this fits on one small GPU.


In [ ]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=16, lora_alpha=32, target_modules="all-linear",
                  lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 4. Fine-tune, then save the adapter

Supervised fine-tuning with `trl`'s `SFTTrainer`. Effective batch size 8 (batch 1 × grad-accum 8) keeps memory low; validation loss is checked each epoch. The adapter is copied to Google Drive the moment training finishes so an idle disconnect can't lose it.


In [ ]:
from trl import SFTTrainer, SFTConfig
cfg = SFTConfig(output_dir="qwen25-3b-jd-lora-v2", num_train_epochs=3,
                per_device_train_batch_size=1, gradient_accumulation_steps=8,
                learning_rate=2e-4, logging_steps=5, eval_strategy="epoch",
                save_strategy="epoch", max_length=2048, dataset_text_field="text", report_to="none")
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds,
                     eval_dataset=val_ds, processing_class=tokenizer)
trainer.train()

# --- save adapter to Drive the moment training finishes ---
model.save_pretrained("qwen25-3b-jd-lora-v2/adapter")
tokenizer.save_pretrained("qwen25-3b-jd-lora-v2/adapter")
!cp -r qwen25-3b-jd-lora-v2/adapter "/content/drive/MyDrive/qwen25-3b-jd-lora-v2-adapter"
print("training + save complete ✅")

## 5. Evaluate — base vs. fine-tuned

The whole point of the project is the **delta**. Using one loaded model, `disable_adapter()` turns the fine-tune *off* to measure the base model, so both are scored under identical decoding and the same field-level metric (`src/evaluation.py`). Each prediction passes through the `JobExtraction` pydantic gate (`validate_prediction`); anything non-conforming counts as a failure.


In [ ]:
from src.evaluation import evaluate, validate_prediction
model.eval()

def generate_raw(jd, max_new_tokens=1024):
    msgs = [{"role":"system","content":EXTRACTION_PROMPT},{"role":"user","content":jd}]
    text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inp = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

def lenient_parse(raw):
    a, b = raw.find("{"), raw.rfind("}")
    try: return json.loads(raw[a:b+1])
    except json.JSONDecodeError: return None

test = [json.loads(l) for l in open("data/processed/test.jsonl") if l.strip()]

def run_eval(tag):
    preds, truths, n_fail = [], [], 0
    for rec in test:
        pred = lenient_parse(generate_raw(df.loc[rec["id"], "job_description"]))
        if pred is not None:
            pred = validate_prediction(pred)          # schema gate -> None if non-conforming
        if pred is None:
            n_fail += 1
            pred = {"required_skills":["__fail__"],"tech_stack":["__fail__"],
                    "seniority":"__invalid__","avg_comp_range":-1}
        preds.append(pred); truths.append(rec)
    res = evaluate(preds, truths); res["valid_json_rate"] = (len(test)-n_fail)/len(test)
    print(tag, res); return res

tuned = run_eval("TUNED   :")
with model.disable_adapter():
    baseline = run_eval("BASELINE:")

## 6. Publish the adapter to the Hugging Face Hub

Model artifacts belong in a registry, not git. Push the ~60 MB LoRA adapter to the Hub so it loads anywhere with `PeftModel.from_pretrained(base, "tkatz123/qwen2.5-3b-job-extraction")` — including the inference service. Log in with a **write** token via the widget (keeps the token out of the notebook).


In [ ]:
!pip install -q -U huggingface_hub
from huggingface_hub import notebook_login
notebook_login()   # paste a WRITE token into the widget

In [ ]:
from huggingface_hub import HfApi, create_repo, whoami

user = whoami()["name"]
REPO = f"{user}/qwen2.5-3b-job-extraction"

create_repo(REPO, repo_type="model", exist_ok=True)
HfApi().upload_folder(
    folder_path="/content/drive/MyDrive/qwen25-3b-jd-lora-v2-adapter",
    repo_id=REPO, repo_type="model",
)
print("pushed ->", f"https://huggingface.co/{REPO}")